# Writeup

**Model Selection & Input Design
Select appropriate ML models (classical or deep learning).
Define scalogram input structure (single-/multi-channel, stacking, reduction).
Justify choices based on data and representation properties.**

This project uses scalograms derived from EEG signals as the main input representations for classification. Each sample corresponds to a single question and is stored as a stacked NumpyArray in the form stacked_full.npy. The notebook uses a stacked scalogram with 4 channels where each channel corresponds to one EEG channel.

The model tested in this project was a simple CNN since scalograms are like images and include a time-frequency representation with spatial structure. CNNs seemed like the first choice when evaluating this type of data because the convolutional filters can learn location specific time-frequency patterns and use this to extract abstract patterns.

The CNN I created was intentionally lightweight to test how a simple CNN would perform. The architecture is comprised of three convolutional blocks as described below.

Conv2d(4 → 16) + ReLU + max pooling
Conv2d(16 → 32) + ReLU + max pooling
Conv2d(32 → 64) + ReLU + adaptive average pooling

The final classification uses a flatten feature vector that has a fully connected layer with 64 hidden units, ReLU activation, dropout (rate of 0.3), and a final linear output (for binary classification).

Because scalogram widths vary, I applied a pad/crop strategy to ensure the CNN was fed scalograms of the same dimensions. The target width can be set to whatever value the user wants; however, I set it to the median width of the dataset. If a scalogram is wider than the target width, it is center-cropped. If it is narrower, a zero-padding is applied symmetrically. Again, this step is necessary because CNN batches require tensors of consistent shape. Finally, normalization is applied using z-score transform.

**Baseline & Training Protocol
Implement simple baselines.
Define train/validation/test strategy (subject-wise vs trial-wise).
Handle class imbalance and tuning procedures.**

The baseline model I create simply makes predictions on a per question basis. Essentially, to set this up, a GroupShuffleSplit with test_size = 0.2 and random_state = 42 was created. The grouping was done by participant_id to ensure all question-level scalograms from the same participant are in the same split. This is important because having question-level scalograms in both the training and test set would introduce leakage. In this simple baseline, class weights were not accounted for; in future models, class weights were taken into account.

Additionally, a baseline model for this problem could be a model that just predicts yes. In this dataset, resulting accuracy would be ~64% and ROC-AUC would be 0.5.


**Model Training & Evaluation
Train models using defined protocol.
Evaluate using appropriate metrics.
Compare performance across models and representations.**

For the evaluation of all models, I computed the following metrics: Accuracy and ROC-AUC. Accuracy measures the proportion of correctly classified samples, while ROC-AUC evaluates the model’s ability to separate the binary class.

While models are trained on question-level scalograms, the ground truth label is defined at the participant level. Thus, evaluation was performed at two levels: question-level performance and participant-level performance.

For the question-level evaluation, each scalogram was treated as an independent sample. The model outputs the logit for each input, which is then converted into a probability using the sigmoid function. Predictions are made by thresholding probabilities at 0.5. Accuracy and ROC-AUC were computed across all question-level samples in the validation set.

For the participant-level evaluation, the predicted probabilities from all questions belonging to the same participant were aggregated into a single prediction. So for each participant, the model produces probabilities for all questions associated with the given participant. These are then aggregated to obtain the participant-level metrics.

For aggregation, I computed across mean aggregation, and median aggregation. In mean aggregation, the probabilities across all questions for a given participant are averaged, while median aggregation uses median probability across the questions. The results of the participant-level probabilities are then thresholding at 0.5 to produce final class predictions.

In the final evaluation, the mean aggregation method was used as the primary metric, while median aggregation was included simply for robustness check.


**Robustness & Generalization
Assess cross-subject generalization.
Evaluate sensitivity to scalogram parameters and input resolution.**

In order to increase robustness and generalization, a Leave-One-Group-Out (LOGO) Cross Validation approach was used. Each group corresponds to a participant (participant_id). In this setup, all the question-level scalograms from a single participant are held out for the evaluation while the model is trained on all the data from the remaining participants. This process is repeated for every participant, so the model is evaluated on every single participant.

This cross validation is extremely critical for classification on the scalograms because models can otherwise learn participant-specific patterns rather than signals related to the actual target. LOGO also ensures that questions for a given participant only appear in either the training or validation set.

Additionally, a weighted loss function was used to address class imbalance. Because the number of participants was not perfectly balanced, training with a standard binary cross-entropy loss could cause the model to just favor the majority class. To mitigate this issue, a weighted binary cross-entropy loss was used where the positive class receives a weighted penalty when misclassified. In this dataset, since the positive class (“yes”) is the majority class, the weighted loss is less than 1, reducing its influence during training and helping balance the contribution of both classes. In LOGO weighted loss was computed at the participant level for each fold. This ensures the model treats both classes equally during optimization.


**Interpretation & Error Analysis
Analyze learned patterns (e.g., saliency or feature importance).
Examine failure cases and misclassification trends.
Deliverables: Report and Code Notebook
Document models, parameters, and experiments, discussions.
Provide a reproducible end-to-end code.**

Results across models were very poor. A simple CNN should be able to extract some signal given proper data. Many models tested performed around 50% accuracy indicating there was little to no signal within my scalograms.

In principle, a CNN should be capable of learning meaningful time-frequency features that scalograms represent. However, the performances I observed indicate that the scalograms either have weak discriminative signals relating to belonging, or that the signal from the scalograms is not sufficiently captured by the model architecture.

There are several factors that may contribute to this result. First, the dataset is relatively small which may limit the model’s ability to learn robust patterns. This may reduce generalizability across subjects. Second, the label is defined at the participant level, while training occurs at the question level. This mismatch could introduce noise into the model training process. Third, scalograms were computed across an entire question instead of using a windowing approach to save time and space complexity. By windowing, there would be more scalograms, and thus more data. Additionally windowing may

An examination of prediction outputs show that many predictions were clustered near the 0.5 decision threshold. This means the model was often uncertain in its classification.

Future work could explore generating new scalograms for training using a windowing approach. Additionally, creating more advanced architectures once baselines start capturing signals could be beneficial.





# Load in Data

In [ ]:
import re
import copy
import pandas as pd
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
ROOT_DIR = Path("/content/drive/MyDrive/Scalograms_numpy_per_question_stacked")
LABELS_CSV = Path("/content/drive/MyDrive/GT1_labels.csv")
assert ROOT_DIR.exists(), f"Can't find: {ROOT_DIR}"
print("Found:", ROOT_DIR)

Found: /content/drive/MyDrive/Scalograms_numpy_per_question_stacked


In [ ]:
# build the pathing of the scalograms given the format of
# ROOT_DIR/
def build_file_dataframe(root_dir):
    rows = []
    for condition_dir in root_dir.iterdir():
        if not condition_dir.is_dir():
            continue
        condition = condition_dir.name  # for the way set up its fixed / control / growth

        for participant_dir in condition_dir.iterdir():
            if not participant_dir.is_dir():
                continue
            participant_id = participant_dir.name

            for question_dir in participant_dir.iterdir():
                if not question_dir.is_dir():
                    continue
                question = question_dir.name
                npy_path = question_dir / "stacked_full.npy"
                if npy_path.exists():
                    rows.append({
                        "condition": condition,
                        "participant_id": str(participant_id),
                        "question": question,
                        "path": str(npy_path)
                    })
    return pd.DataFrame(rows)

# view
files_df = build_file_dataframe(ROOT_DIR)
print("Number of examples:", len(files_df))
files_df.head()

Number of examples: 923


,condition,participant_id,question,path
0,control,67c26b6a6aed7a8612189b14,Question01,/content/drive/MyDrive/Scalograms_numpy_per_qu...
1,control,67c26b6a6aed7a8612189b14,Question02,/content/drive/MyDrive/Scalograms_numpy_per_qu...
2,control,67c26b6a6aed7a8612189b14,Question03,/content/drive/MyDrive/Scalograms_numpy_per_qu...
3,control,67c26b6a6aed7a8612189b14,Question04,/content/drive/MyDrive/Scalograms_numpy_per_qu...
4,control,67c26b6a6aed7a8612189b14,Question05,/content/drive/MyDrive/Scalograms_numpy_per_qu...


### Adding Labels

In [ ]:
labels_df = pd.read_csv(LABELS_CSV)

print(labels_df.head())
print(labels_df.columns)

                 student_id  GT1
0  67c269abbbbc97c24e64d8dd    1
1  67c26b6a6aed7a8612189b14    0
2  67c26f8829d462ae823bb2c2    1
3  67c26fde6aed7a861218a3cb    1
4  67c271ffe81e2cd37b70360b    1
Index(['student_id', 'GT1'], dtype='object')


In [ ]:
files_df["participant_id"] = files_df["participant_id"].astype(str)
labels_df["student_id"] = labels_df["student_id"].astype(str)

# Keep only the columns we need
labels_df = labels_df[["student_id", "GT1"]].copy()

# Merge
df = files_df.merge(
    labels_df,
    left_on="participant_id",
    right_on="student_id",
    how="inner"
)

# Rename GT1 to y just for conveinence
df = df.rename(columns={"GT1": "y"})

print(df.head())
print("Merged examples:", len(df))
print(df["y"].value_counts(dropna=False))

  condition            participant_id    question  \
0   control  67c26b6a6aed7a8612189b14  Question01   
1   control  67c26b6a6aed7a8612189b14  Question02   
2   control  67c26b6a6aed7a8612189b14  Question03   
3   control  67c26b6a6aed7a8612189b14  Question04   
4   control  67c26b6a6aed7a8612189b14  Question05   

                                                path  \
0  /content/drive/MyDrive/Scalograms_numpy_per_qu...   
1  /content/drive/MyDrive/Scalograms_numpy_per_qu...   
2  /content/drive/MyDrive/Scalograms_numpy_per_qu...   
3  /content/drive/MyDrive/Scalograms_numpy_per_qu...   
4  /content/drive/MyDrive/Scalograms_numpy_per_qu...   

                 student_id  y  
0  67c26b6a6aed7a8612189b14  0  
1  67c26b6a6aed7a8612189b14  0  
2  67c26b6a6aed7a8612189b14  0  
3  67c26b6a6aed7a8612189b14  0  
4  67c26b6a6aed7a8612189b14  0  
Merged examples: 878
y
1    558
0    320
Name: count, dtype: int64


In [ ]:
df = df.dropna(subset=["y"]).copy()
df["y"] = df["y"].astype(int)

print(df["y"].value_counts())

y
1    558
0    320
Name: count, dtype: int64


### Defining Baseline

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42) # create the group shuffle split
train_idx, val_idx = next(gss.split(df, y=df["y"], groups=df["participant_id"]))

# create training and validation dataframes
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

# print info
print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Train participants:", train_df["participant_id"].nunique())
print("Val participants:", val_df["participant_id"].nunique())

Train size: 686
Val size: 192
Train participants: 22
Val participants: 6


In [ ]:
class ScalogramDataset(Dataset):
    # just initialization, been using median target width again as default
    def __init__(self, dataframe, target_width=2465):
        self.df = dataframe.reset_index(drop=True)
        self.target_width = target_width

    # simple method to return length of dataframe
    def __len__(self):
        return len(self.df)

    # ensuring all scalograms have identical width for CNN
    def _pad_or_crop(self, x):
        # x shape: (4, H, W)
        current_width = x.shape[2] # current width

        # case where already target
        if current_width == self.target_width:
            return x
        # if scalogram larger than crop it (I do get middle) NOTE: may change this later not sure middle makes sense
        elif current_width > self.target_width:
            # find left and right cropping places
            start = (current_width - self.target_width) // 2
            end = start + self.target_width
            return x[:, :, start:end]
        # Scalogram too short then pad it with 0s on both front and end
        else:
            # find left and right padding places
            pad_total = self.target_width - current_width
            pad_left = pad_total // 2
            pad_right = pad_total - pad_left

            x_padded = np.pad(
                x,
                pad_width=((0, 0), (0, 0), (pad_left, pad_right)),
                mode="constant",
                constant_values=0
            )
            return x_padded

    # get me a sample
    def __getitem__(self, idx):
        # grab the row
        row = self.df.iloc[idx]
        x = np.load(row["path"]).astype(np.float32) # load the scalogram array

        # safety check, if we reach here then did something wrong (expecting (C, H, W) where C = 4)
        if x.ndim != 3:
            raise ValueError(f"Expected 3D array, got shape {x.shape}")

        # need the dimensions (number of channels) to be size 4
        if x.shape[0] == 4:
            pass
        # if in the incorrect format, turn into (4, H , W)
        elif x.shape[-1] == 4:
            x = np.transpose(x, (2, 0, 1))
        # case where wrong dimension size (wrong number of channels)
        else:
            raise ValueError(f"Unexpected shape {x.shape}; expected channel dim of size 4")

        x = self._pad_or_crop(x) # call the pad/crop method

        # normalize after pad/crop
        x = (x - x.mean()) / (x.std() + 1e-8)
        y = np.float32(row["y"])
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

In [ ]:
# used because CNN requires same size for inputs

widths = []

for p in df["path"]:
    arr = np.load(p)
    if arr.shape[0] == 4:
        widths.append(arr.shape[2])
    elif arr.shape[-1] == 4:
        widths.append(arr.shape[1])
    else:
        print("Unexpected shape:", arr.shape)

print("Min width:", min(widths))
print("Max width:", max(widths))
print("Median width:", np.median(widths))

Min width: 24
Max width: 381125
Median width: 2469.0


In [ ]:
# median was chosen arbitrarily, just wanted to make sure that we were pretty close to what represented the data.

target_width = int(np.median(widths))
train_dataset = ScalogramDataset(train_df, target_width=target_width)
val_dataset = ScalogramDataset(val_df, target_width=target_width)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)
print(target_width)

2469


In [ ]:
# just a regular CNN for extracting information from my scalograms
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=4, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x.squeeze(1)

## Baseline (Per Question)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = SimpleCNN().to(device)

y_train = train_df["y"].values
num_pos = (y_train == 1).sum()
num_neg = (y_train == 0).sum()

print("num_pos:", num_pos, "num_neg:", num_neg)

# good in practice, but dont really need check. Adjusts for class distribution
if num_pos > 0:
    pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
else:
    criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Using device: cuda
num_pos: 494 num_neg: 192


In [ ]:
# function for training one full pass (epoch) over the training data
# takes in model: the model, loader: DataLoader that contains the question level scalograms, criterion: the loss function used, optimzer: optimizer used, device: cuda
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train() # train
    total_loss = 0.0

    # loop thru batches
    for X, y in loader:
        # want on GPU or else it will take forever
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad() # clear old gradients
        logits = model(X) # get logits
        loss = criterion(logits, y) # get loss
        loss.backward() # get gradients
        optimizer.step() # apply gradients

        total_loss += loss.item() * X.size(0) # accumulate total loss across samples (loss.item() is avg over batch)

    return total_loss / len(loader.dataset) # avg loss per sample over given epoch

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval() # model eval mode
    total_loss = 0.0
    # defining metrics
    all_probs = []
    all_preds = []
    all_targets = []
    for X, y in loader:
        # Move to GPU
        X = X.to(device)
        y = y.to(device)

        logits = model(X) # logits
        loss = criterion(logits, y) # compute loss

        probs = torch.sigmoid(logits) # calculate probabilities
        preds = (probs >= 0.5).float() # make actual predictions now

        total_loss += loss.item() * X.size(0) # accumulate total loss across samples (loss.item() is avg over batch)

        # store data, use CPU for numpy
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    loss = total_loss / len(loader.dataset) # compute final loss
    acc = accuracy_score(all_targets, all_preds) # % of correct classification of questions

    try:
        auc = roc_auc_score(all_targets, all_probs) # compute roc_auc
    except ValueError:
        auc = np.nan

    return loss, acc, auc

In [ ]:
X, y = next(iter(train_loader))
print(X.shape, y.shape)

torch.Size([16, 4, 64, 2469]) torch.Size([16])


In [ ]:
num_epochs = 10 # started with 10, can change to be larger if wanted

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_auc = evaluate(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val AUC: {val_auc:.4f}"
    )

Epoch 1/10 | Train Loss: 0.3743 | Val Loss: 0.5952 | Val Acc: 0.3385 | Val AUC: 0.3322
Epoch 2/10 | Train Loss: 0.3468 | Val Loss: 0.6824 | Val Acc: 0.2500 | Val AUC: 0.2158
Epoch 3/10 | Train Loss: 0.3412 | Val Loss: 0.7622 | Val Acc: 0.3281 | Val AUC: 0.4244
Epoch 4/10 | Train Loss: 0.3256 | Val Loss: 0.7715 | Val Acc: 0.3281 | Val AUC: 0.2700
Epoch 5/10 | Train Loss: 0.3043 | Val Loss: 0.7642 | Val Acc: 0.2760 | Val AUC: 0.2173
Epoch 6/10 | Train Loss: 0.2771 | Val Loss: 0.7250 | Val Acc: 0.4896 | Val AUC: 0.5232
Epoch 7/10 | Train Loss: 0.2620 | Val Loss: 0.6966 | Val Acc: 0.4375 | Val AUC: 0.1866
Epoch 8/10 | Train Loss: 0.2458 | Val Loss: 0.5395 | Val Acc: 0.5677 | Val AUC: 0.5485
Epoch 9/10 | Train Loss: 0.2695 | Val Loss: 1.0002 | Val Acc: 0.3750 | Val AUC: 0.4183
Epoch 10/10 | Train Loss: 0.2245 | Val Loss: 0.5140 | Val Acc: 0.5469 | Val AUC: 0.5164


## Aggregate (Participant Level)

In [ ]:
# Pytorch Dataset for LOGO approach
class TrainScalogramDataset(Dataset):
    # just initialization, been using median target width again as default
    def __init__(self, dataframe, target_width=2465):
        self.df = dataframe.reset_index(drop=True)
        self.target_width = target_width

    # simple method to return length of dataframe
    def __len__(self):
        return len(self.df)

    # ensuring all scalograms have identical width for CNN
    def _pad_or_crop(self, x):
        current_width = x.shape[2] # current width

        # case where already target
        if current_width == self.target_width:
            return x
        # if scalogram larger than crop it (I do get middle) NOTE: may change this later not sure middle makes sense
        elif current_width > self.target_width:
            # find left and right cropping places
            start = (current_width - self.target_width) // 2
            end = start + self.target_width
            return x[:, :, start:end] # actual crop
        # Scalogram too short then pad it with 0s on both front and end
        else:
            # find left and right padding places
            pad_total = self.target_width - current_width
            pad_left = pad_total // 2
            pad_right = pad_total - pad_left
            # do the actual padding
            return np.pad(
                x,
                pad_width=((0, 0), (0, 0), (pad_left, pad_right)),
                mode="constant",
                constant_values=0
            )

    # get me a sample
    def __getitem__(self, idx):
        # grab the row
        row = self.df.iloc[idx]
        x = np.load(row["path"]).astype(np.float32) # load the scalogram array

        # safety check, if we reach here then did something wrong (expecting (C, H, W) where C = 4)
        if x.ndim != 3:
            raise ValueError(f"Expected 3D array, got shape {x.shape}")

        # need the dimensions (number of channels) to be size 4
        if x.shape[0] == 4:
            pass
        # if in the incorrect format, turn into (4, H , W)
        elif x.shape[-1] == 4:
            x = np.transpose(x, (2, 0, 1))
        # case where wrong dimension size (wrong number of channels)
        else:
            raise ValueError(f"Unexpected shape {x.shape}; expected channel dim of size 4")

        # per-channel normalization
        x_mean = x.mean(axis=(1, 2), keepdims=True)
        x_std = x.std(axis=(1, 2), keepdims=True) + 1e-8
        x = (x - x_mean) / x_std

        x = self._pad_or_crop(x) # call the pad/crop method

        # convertign to pytorch tensor
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(row["y"], dtype=torch.float32)

        return x, y


# identical as above except for what is returned. This is used for validation dataset
class EvalScalogramDataset(Dataset):
    # just initialization, been using median target width again as default
    def __init__(self, dataframe, target_width=2465):
        self.df = dataframe.reset_index(drop=True)
        self.target_width = target_width

    # simple method to return length of dataframe
    def __len__(self):
        return len(self.df)

    # ensuring all scalograms have identical width for CNN
    def _pad_or_crop(self, x):
        current_width = x.shape[2] # current width

        # case where already target
        if current_width == self.target_width:
            return x
        # if scalogram larger than crop it (I do get middle) NOTE: may change this later not sure middle makes sense
        elif current_width > self.target_width:
            # find left and right cropping places
            start = (current_width - self.target_width) // 2
            end = start + self.target_width
            return x[:, :, start:end]
        # Scalogram too short then pad it with 0s on both front and end
        else:
            # find left and right padding places
            pad_total = self.target_width - current_width
            pad_left = pad_total // 2
            pad_right = pad_total - pad_left
            # do the actual padding
            return np.pad(
                x,
                pad_width=((0, 0), (0, 0), (pad_left, pad_right)),
                mode="constant",
                constant_values=0
            )

    # get me a sample
    def __getitem__(self, idx):
        # grab the row
        row = self.df.iloc[idx]
        x = np.load(row["path"]).astype(np.float32) # load the scalogram array

        # safety check, if we reach here then did something wrong (expecting (C, H, W) where C = 4)
        if x.ndim != 3:
            raise ValueError(f"Expected 3D array, got shape {x.shape}")

        # need the dimensions (number of channels) to be size 4
        if x.shape[0] == 4:
            pass
        # if in the incorrect format, turn into (4, H , W)
        elif x.shape[-1] == 4:
            x = np.transpose(x, (2, 0, 1))
        # case where wrong dimension size (wrong number of channels)
        else:
            raise ValueError(f"Unexpected shape {x.shape}; expected channel dim of size 4")

        # per-channel normalization
        x_mean = x.mean(axis=(1, 2), keepdims=True)
        x_std = x.std(axis=(1, 2), keepdims=True) + 1e-8
        x = (x - x_mean) / x_std

        x = self._pad_or_crop(x) # call the pad/crop method

        # convertign to pytorch tensor
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(row["y"], dtype=torch.float32)

        # difference from trainer, returns participant_id, condition, and question
        return (
            x,
            y,
            str(row["participant_id"]),
            str(row["condition"]),
            str(row["question"])
        )

In [ ]:
# not track graidents to save memory and computation
@torch.no_grad()
def predict_aggregate(model, loader, device, threshold=0.5):
    model.eval()

    # init to collect metrics
    all_probs = []
    all_targets = []
    all_participants = []
    all_conditions = []
    all_questions = []

    # iterate thru the loader. X: batch of question scalgorams, y: labels, participant_ids: id, conditions: control/fixed/growth, questions: question identifer
    for X, y, participant_ids, conditions, questions in loader:
        X = X.to(device)
        logits = model(X) # outputs a logit
        probs = torch.sigmoid(logits).cpu().numpy() # map logits to 0, 1, by going back to CPU cause we were on GPU

        # storing results
        all_probs.extend(probs)
        all_targets.extend(y.numpy())
        all_participants.extend(participant_ids)
        all_conditions.extend(conditions)
        all_questions.extend(questions)

    # create the dataframe of raw preds for questions
    pred_df = pd.DataFrame({
        "participant_id": all_participants,
        "condition": all_conditions,
        "question": all_questions,
        "prob": all_probs,
        "y": all_targets
    })

    # from questions, aggregate for participant because we care about particpant preds
    # avg/median accross quetions for participant to get overall scores
    participant_df = (
        pred_df.groupby("participant_id", as_index=False)
        .agg(
            prob_mean=("prob", "mean"),
            prob_median=("prob", "median"),
            n_questions=("prob", "size"),
            y=("y", "first")
        )
    )

    # Getting actual label
    participant_df["pred_mean"] = (participant_df["prob_mean"] >= threshold).astype(int)
    participant_df["pred_median"] = (participant_df["prob_median"] >= threshold).astype(int)

    participant_acc_mean = accuracy_score(participant_df["y"], participant_df["pred_mean"])

    # added a safeguard to break; kind of annoying tho
    try:
        participant_auc_mean = roc_auc_score(participant_df["y"], participant_df["prob_mean"])
    except ValueError:
        participant_auc_mean = np.nan

    return pred_df, participant_df, participant_acc_mean, participant_auc_mean




In [ ]:
# function for training one full pass (epoch) over the training data
# takes in model: the model, loader: DataLoader that contains the question level scalograms, criterion: the loss function used, optimzer: optimizer used, device: cuda
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train() # train
    total_loss = 0.0

    # loop thru batches
    for X, y in loader:
        # want on GPU or else it will take forever
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad() # clear old gradients
        logits = model(X) # get logits
        loss = criterion(logits, y) # get loss
        loss.backward() # get gradients
        optimizer.step() # apply gradients

        total_loss += loss.item() * X.size(0) # accumulate total loss across samples (loss.item() is avg over batch)

    return total_loss / len(loader.dataset) # avg loss per sample over given epoch

In [ ]:
@torch.no_grad() # good best practice
# Function to evaulate model on an individual question (one scalogram because one scalogram per question)
def evaluate_question_level(model, loader, criterion, device):
    model.eval() # model eval mode
    total_loss = 0.0
    # defining metrics
    all_probs = []
    all_preds = []
    all_targets = []

    for X, y in loader:
        # Move to GPU
        X = X.to(device)
        y = y.to(device)

        logits = model(X) # logits
        loss = criterion(logits, y) # compute loss

        probs = torch.sigmoid(logits) # calculate probabilities
        preds = (probs >= 0.5).float() # make actual predictions now

        total_loss += loss.item() * X.size(0) # accumulate total loss across samples (loss.item() is avg over batch)

        # store data, use CPU for numpy
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    loss = total_loss / len(loader.dataset) # compute final loss
    acc = accuracy_score(all_targets, all_preds) # % of correct classification of questions

    try:
        auc = roc_auc_score(all_targets, all_probs) # compute roc_auc
    except ValueError:
        auc = np.nan # gonna happen

    return loss, acc, auc

In [ ]:
# # creating dataset objects (PyTorch Dataset)
# train_dataset = ScalogramDataset(train_df)
# val_dataset = ScalogramDataset(val_df)

# # creating training/val DataLoader
# train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [ ]:
# creating dataset objects (PyTorch Dataset)
train_dataset = TrainScalogramDataset(train_df)
val_dataset_q = TrainScalogramDataset(val_df)
val_dataset_agg = EvalScalogramDataset(val_df)

# creating training/val DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader_q = DataLoader(val_dataset_q, batch_size=16, shuffle=False, num_workers=0)
val_loader_agg = DataLoader(val_dataset_agg, batch_size=16, shuffle=False, num_workers=0)

In [ ]:
# epochs want to train
num_epochs = 15

# do the actual training
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # get metrics
    val_q_loss, val_q_acc, val_q_auc = evaluate_question_level(model, val_loader_q, criterion, device)
    _, participant_df, val_p_acc, val_p_auc = predict_aggregate(model, val_loader_agg, device)

    # print the metrics for the epoch
    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Question Acc: {val_q_acc:.4f} | "
        f"Val Question AUC: {val_q_auc:.4f} | "
        f"Val Participant Acc: {val_p_acc:.4f} | "
        f"Val Participant AUC: {val_p_auc:.4f}"
    )

Epoch 1/15 | Train Loss: 0.2320 | Val Question Acc: 0.5052 | Val Question AUC: 0.6271 | Val Participant Acc: 0.5000 | Val Participant AUC: 0.6250
Epoch 2/15 | Train Loss: 0.2150 | Val Question Acc: 0.4323 | Val Question AUC: 0.6942 | Val Participant Acc: 0.3333 | Val Participant AUC: 0.8750
Epoch 3/15 | Train Loss: 0.2143 | Val Question Acc: 0.5990 | Val Question AUC: 0.6755 | Val Participant Acc: 0.6667 | Val Participant AUC: 0.8750
Epoch 4/15 | Train Loss: 0.2191 | Val Question Acc: 0.6198 | Val Question AUC: 0.7898 | Val Participant Acc: 0.5000 | Val Participant AUC: 0.7500
Epoch 5/15 | Train Loss: 0.2032 | Val Question Acc: 0.5469 | Val Question AUC: 0.5956 | Val Participant Acc: 0.6667 | Val Participant AUC: 0.6250
Epoch 6/15 | Train Loss: 0.1797 | Val Question Acc: 0.5469 | Val Question AUC: 0.6980 | Val Participant Acc: 0.5000 | Val Participant AUC: 0.7500
Epoch 7/15 | Train Loss: 0.1793 | Val Question Acc: 0.6198 | Val Question AUC: 0.7473 | Val Participant Acc: 0.5000 | Val Pa

In [ ]:
# Just some extra Stuff
print("Train participants by label:")
print(train_df.groupby("participant_id")["y"].first().value_counts())

print("\nVal participants by label:")
print(val_df.groupby("participant_id")["y"].first().value_counts())

pred_df, participant_df, val_p_acc, val_p_auc = predict_aggregate(model, val_loader_agg, device)

print("\nParticipant predictions:")
print(participant_df.sort_values("prob_mean"))

Train participants by label:
y
1    16
0     6
Name: count, dtype: int64

Val participants by label:
y
0    4
1    2
Name: count, dtype: int64

Participant predictions:
             participant_id  prob_mean  prob_median  n_questions    y  \
3  67c7815abe53b42f18bcbb45   0.647402     0.707457           32  1.0   
1  67c777d88cb8fc5788663844   0.677571     0.675499           32  0.0   
0  67c26b6a6aed7a8612189b14   0.772978     0.876111           32  0.0   
4  67d0a983ecca3a423861ccbd   0.821660     0.937731           32  0.0   
2  67c77911ff18ef1bd5d4ca8a   0.983570     0.991882           32  1.0   
5  680ac76059c825dbf7c315de   0.994109     0.997121           32  0.0   

   pred_mean  pred_median  
3          1            1  
1          1            1  
0          1            1  
4          1            1  
2          1            1  
5          1            1  


## Applying LOGO (Cross Validation)

In [ ]:
# Pytorch Dataset for LOGO approach
class TrainScalogramDataset(Dataset):
    # just initialization, been using median target width again as default
    def __init__(self, dataframe, target_width=2465):
        self.df = dataframe.reset_index(drop=True)
        self.target_width = target_width

    # simple method to return length of dataframe
    def __len__(self):
        return len(self.df)

    # ensuring all scalograms have identical width for CNN
    def _pad_or_crop(self, x):
        current_width = x.shape[2] # current width

        # case where already target
        if current_width == self.target_width:
            return x
        # if scalogram larger than crop it (I do get middle) NOTE: may change this later not sure middle makes sense
        elif current_width > self.target_width:
            # find left and right cropping places
            start = (current_width - self.target_width) // 2
            end = start + self.target_width
            return x[:, :, start:end] # actual crop
        # Scalogram too short then pad it with 0s on both front and end
        else:
            # find left and right padding places
            pad_total = self.target_width - current_width
            pad_left = pad_total // 2
            pad_right = pad_total - pad_left
            # do the actual padding
            return np.pad(
                x,
                pad_width=((0, 0), (0, 0), (pad_left, pad_right)),
                mode="constant",
                constant_values=0
            )

    # get me a sample
    def __getitem__(self, idx):
        # grab the row
        row = self.df.iloc[idx]
        x = np.load(row["path"]).astype(np.float32) # load the scalogram array

        # safety check, if we reach here then did something wrong (expecting (C, H, W) where C = 4)
        if x.ndim != 3:
            raise ValueError(f"Expected 3D array, got shape {x.shape}")

        # need the dimensions (number of channels) to be size 4
        if x.shape[0] == 4:
            pass
        # if in the incorrect format, turn into (4, H , W)
        elif x.shape[-1] == 4:
            x = np.transpose(x, (2, 0, 1))
        # case where wrong dimension size (wrong number of channels)
        else:
            raise ValueError(f"Unexpected shape {x.shape}; expected channel dim of size 4")

        # per-channel normalization
        x_mean = x.mean(axis=(1, 2), keepdims=True)
        x_std = x.std(axis=(1, 2), keepdims=True) + 1e-8
        x = (x - x_mean) / x_std

        x = self._pad_or_crop(x) # call the pad/crop method

        # convertign to pytorch tensor
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(row["y"], dtype=torch.float32)

        return x, y


# identical as above except for what is returned. This is used for validation dataset
class EvalScalogramDataset(Dataset):
    # just initialization, been using median target width again as default
    def __init__(self, dataframe, target_width=2465):
        self.df = dataframe.reset_index(drop=True)
        self.target_width = target_width

    # simple method to return length of dataframe
    def __len__(self):
        return len(self.df)

    # ensuring all scalograms have identical width for CNN
    def _pad_or_crop(self, x):
        current_width = x.shape[2] # current width

        # case where already target
        if current_width == self.target_width:
            return x
        # if scalogram larger than crop it (I do get middle) NOTE: may change this later not sure middle makes sense
        elif current_width > self.target_width:
            # find left and right cropping places
            start = (current_width - self.target_width) // 2
            end = start + self.target_width
            return x[:, :, start:end]
        # Scalogram too short then pad it with 0s on both front and end
        else:
            # find left and right padding places
            pad_total = self.target_width - current_width
            pad_left = pad_total // 2
            pad_right = pad_total - pad_left
            # do the actual padding
            return np.pad(
                x,
                pad_width=((0, 0), (0, 0), (pad_left, pad_right)),
                mode="constant",
                constant_values=0
            )

    # get me a sample
    def __getitem__(self, idx):
        # grab the row
        row = self.df.iloc[idx]
        x = np.load(row["path"]).astype(np.float32) # load the scalogram array

        # safety check, if we reach here then did something wrong (expecting (C, H, W) where C = 4)
        if x.ndim != 3:
            raise ValueError(f"Expected 3D array, got shape {x.shape}")

        # need the dimensions (number of channels) to be size 4
        if x.shape[0] == 4:
            pass
        # if in the incorrect format, turn into (4, H , W)
        elif x.shape[-1] == 4:
            x = np.transpose(x, (2, 0, 1))
        # case where wrong dimension size (wrong number of channels)
        else:
            raise ValueError(f"Unexpected shape {x.shape}; expected channel dim of size 4")

        # per-channel normalization
        x_mean = x.mean(axis=(1, 2), keepdims=True)
        x_std = x.std(axis=(1, 2), keepdims=True) + 1e-8
        x = (x - x_mean) / x_std

        x = self._pad_or_crop(x) # call the pad/crop method

        # convertign to pytorch tensor
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(row["y"], dtype=torch.float32)

        # difference from trainer, returns participant_id, condition, and question
        return (
            x,
            y,
            str(row["participant_id"]),
            str(row["condition"]),
            str(row["question"])
        )

In [ ]:
# function for training one full pass (epoch) over the training data. This is for LOGO, essentially the same as the one used for Participant level, but added for clairity
# takes in model: the model, loader: DataLoader that contains the question level scalograms, criterion: the loss function used, optimzer: optimizer used, device: cuda
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0 # train

    # loop thru batches
    for X, y in loader:
        # want on GPU or else it will take forever
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad() # clear old gradients
        logits = model(X) # get logits
        loss = criterion(logits, y) # get loss
        loss.backward() # get gradients
        optimizer.step() # apply gradients

        total_loss += loss.item() * X.size(0) # accumulate total loss across samples (loss.item() is avg over batch)

    return total_loss / len(loader.dataset) # avg loss per sample over given epoch


@torch.no_grad() # good best practice
# This is for LOGO, essentially the same as the one used for Participant level, but added for clairity
def evaluate_question_level(model, loader, criterion, device, threshold=0.5):
    model.eval() # model eval mode
    # defining metrics
    total_loss = 0.0
    all_probs = []
    all_preds = []
    all_targets = []

    for X, y in loader:
        # Move to GPU
        X = X.to(device)
        y = y.to(device)

        logits = model(X) # logits
        loss = criterion(logits, y) # compute loss

        probs = torch.sigmoid(logits) # calculate probabilities
        preds = (probs >= threshold).float() # make actual predictions now (now uses a threshold in case I want to change)

        total_loss += loss.item() * X.size(0) # accumulate total loss across samples (loss.item() is avg over batch)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    loss = total_loss / len(loader.dataset) # compute final loss
    acc = accuracy_score(all_targets, all_preds) # % of correct classification of questions

    try:
        auc = roc_auc_score(all_targets, all_probs) # compute roc_auc
    except ValueError:
        auc = np.nan

    return loss, acc, auc


@torch.no_grad() # best practice
# Takes in model, the DataLaoder, the device: cuda/cpu, and a threshold for class predicitions -> computes participant level metrics
def predict_aggregate(model, loader, device, threshold=0.5):
    model.eval() # model eval mode

    # defining metrics
    all_probs = []
    all_targets = []
    all_participants = []
    all_conditions = []
    all_questions = []

    # iterate thru the loader. X: batch of question scalgorams, y: labels, participant_ids: id, conditions: control/fixed/growth, questions: question identifer
    for X, y, participant_ids, conditions, questions in loader:
        # Move to GPU
        X = X.to(device)
        logits = model(X) # outputs a logit logits
        probs = torch.sigmoid(logits).cpu().numpy() # map logits to 0, 1, by going back to CPU cause we were on GPU

        # store the outputs for the current batch
        all_probs.extend(probs)
        all_targets.extend(y.numpy())
        all_participants.extend(participant_ids)
        all_conditions.extend(conditions)
        all_questions.extend(questions)

    # Question level predicition dataframe
    pred_df = pd.DataFrame({
        "participant_id": all_participants,
        "condition": all_conditions,
        "question": all_questions,
        "prob": all_probs,
        "y": all_targets
    })

    # from questions, aggregate for participant because we care about particpant preds
    # avg/median accross quetions for participant to get overall scores
    participant_df = (
        pred_df.groupby("participant_id", as_index=False)
        .agg(
            prob_mean=("prob", "mean"),
            prob_median=("prob", "median"),
            n_questions=("prob", "size"),
            y=("y", "first")
        )
    )

    # Getting actual label
    participant_df["pred_mean"] = (participant_df["prob_mean"] >= threshold).astype(int)
    participant_df["pred_median"] = (participant_df["prob_median"] >= threshold).astype(int)

    # Getting actual metrics for accuracy per participant
    p_acc_mean = accuracy_score(participant_df["y"], participant_df["pred_mean"])
    p_acc_median = accuracy_score(participant_df["y"], participant_df["pred_median"])

    try:
        p_auc_mean = roc_auc_score(participant_df["y"], participant_df["prob_mean"]) # roc_auc
    except ValueError:
        p_auc_mean = np.nan

    try:
        p_auc_median = roc_auc_score(participant_df["y"], participant_df["prob_median"]) # roc_auc for median
    except ValueError:
        p_auc_median = np.nan

    return pred_df, participant_df, p_acc_mean, p_auc_mean, p_acc_median, p_auc_median

In [ ]:
# function to build PyTorch dataloaders given the training data, validation data, and batchsize
def build_loaders(train_df, val_df, batch_size=16):
    train_dataset = TrainScalogramDataset(train_df) # create training dataframe
    val_dataset_q = TrainScalogramDataset(val_df) # uses train bc we only need question level. Creating question level validation dataframe
    val_dataset_agg = EvalScalogramDataset(val_df) # uses eval bc we need to aggregate for participant level (this datset gives us extra info of participant_id, condition, and question)

    # create the training loader and validation loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader_q = DataLoader(val_dataset_q, batch_size=batch_size, shuffle=False, num_workers=0) # again question level
    val_loader_agg = DataLoader(val_dataset_agg, batch_size=batch_size, shuffle=False, num_workers=0) # participant level

    return train_loader, val_loader_q, val_loader_agg


# Builds a loss function with class weights. Takes in the training dataset and the device (cuda or cpu)
def make_weighted_loss_from_participants(train_df, device):
    train_participant_labels = train_df.groupby("participant_id")["y"].first() # extract participant labels (labels are same for all question for a given participant)

    # counts number of participant in yes or no in the current training fold
    n_pos = (train_participant_labels == 1).sum()
    n_neg = (train_participant_labels == 0).sum()

    if n_pos == 0 or n_neg == 0:
        return None, n_pos, n_neg

    # penalize missing minority class more
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    return criterion, n_pos, n_neg

# Train on all outer-train participants (participants that is not the "left out") for a fixed number of epochs.
def train_fixed_epoch_outer_model(outer_train_df, device, fixed_epochs=5, batch_size=16, learning_rate=1e-3):
    criterion, n_pos, n_neg = make_weighted_loss_from_participants(outer_train_df, device) # create weighted loss

    # occurs if there is only one class in the training set (unlikely)
    if criterion is None:
        raise ValueError("Outer training split has only one class.")

    # build the training dataset and loader
    train_dataset = TrainScalogramDataset(outer_train_df)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    # initialize model
    model = SimpleCNN().to(device)
    # use adam optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # tracking losses in the epochs
    epoch_losses = []
    # do fixed_epochs epochs
    for epoch in range(fixed_epochs):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device) # train a singular epoch
        epoch_losses.append(train_loss) # append training loss to the list
        print(f"  Epoch {epoch+1}/{fixed_epochs} | train_loss={train_loss:.4f}")

    return model, criterion, n_pos, n_neg, epoch_losses

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logo = LeaveOneGroupOut() # defining the LOGO

# can change hyper params (played around with these)
fixed_epochs = 5
batch_size = 16
learning_rate = 1e-3

# groups needed in order to tell LOGO how to split
groups = df["participant_id"].values

# Init to collect results accross folds
all_question_rows = []
all_participant_rows = []
outer_fold_rows = []


# Go through for each participant
for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(logo.split(df, df["y"], groups=groups), start=1):
    # build train and test dataframes for given fold
    outer_train_df = df.iloc[outer_train_idx].reset_index(drop=True)
    outer_test_df = df.iloc[outer_test_idx].reset_index(drop=True)

    held_out_participant = outer_test_df["participant_id"].iloc[0] # this is just identifying heldout participant

    # tests to make sure that no leakage (because leakage would ruin results)
    assert set(outer_train_df["participant_id"]).isdisjoint(set(outer_test_df["participant_id"]))
    assert set(outer_train_df["path"]).isdisjoint(set(outer_test_df["path"]))

    # print info
    print(f"\n{'='*80}")
    print(f"Outer fold {outer_fold} | held-out participant: {held_out_participant}")

    # Train on all outer-train participants for a fixed number of epochs
    model, criterion, n_pos, n_neg, epoch_losses = train_fixed_epoch_outer_model(
        outer_train_df=outer_train_df,
        device=device,
        fixed_epochs=fixed_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate
    )

    # class counts
    print(f"Outer-train participant counts -> pos: {n_pos}, neg: {n_neg}")

    # Evaluate once on the held-out participant
    test_dataset_q = TrainScalogramDataset(outer_test_df)
    test_dataset_agg = EvalScalogramDataset(outer_test_df)

    # build held out loaders for ...
    test_loader_q = DataLoader(test_dataset_q, batch_size=batch_size, shuffle=False, num_workers=0) # question level
    test_loader_agg = DataLoader(test_dataset_agg, batch_size=batch_size, shuffle=False, num_workers=0) # participant level

    # Question level evaluation
    test_q_loss, test_q_acc, test_q_auc = evaluate_question_level(model, test_loader_q, criterion, device)

    # particiapnt level aggregation
    pred_df, participant_df, p_acc_mean, p_auc_mean, p_acc_median, p_auc_median = predict_aggregate(
        model, test_loader_agg, device
    )

    # add in fold metadata
    pred_df["outer_fold"] = outer_fold
    participant_df["outer_fold"] = outer_fold
    participant_df["held_out_participant"] = held_out_participant
    participant_df["fixed_epochs"] = fixed_epochs

    # store predicition at the question and participant level
    all_question_rows.append(pred_df)
    all_participant_rows.append(participant_df)

    # store summary of fold
    outer_fold_rows.append({
        "outer_fold": outer_fold,
        "held_out_participant": held_out_participant,
        "fixed_epochs": fixed_epochs,
        "final_train_loss": epoch_losses[-1],
        "test_q_loss": test_q_loss,
        "test_q_acc": test_q_acc,
        "test_q_auc": test_q_auc,   # usually NaN for one held-out participant
        "test_p_acc_mean": p_acc_mean,
        "test_p_acc_median": p_acc_median
    })

# Combine all outer-fold predictions
question_pred_df = pd.concat(all_question_rows, ignore_index=True)
participant_pred_df = pd.concat(all_participant_rows, ignore_index=True)
outer_summary_df = pd.DataFrame(outer_fold_rows)

print("\nOuter fold summary:")
print(outer_summary_df)

# Mean question level metrics across held-out folds
print("\nMean question-level metrics across outer folds:")
print(outer_summary_df[["test_q_loss", "test_q_acc"]].mean(numeric_only=True))

# Mean participant level accuracy across held-out folds
print("\nMean participant-level accuracy across outer folds:")
print(outer_summary_df[["test_p_acc_mean", "test_p_acc_median"]].mean(numeric_only=True))

# Final participant level metrics across all held-out participants
overall_p_acc_mean = accuracy_score(participant_pred_df["y"], participant_pred_df["pred_mean"])
overall_p_acc_median = accuracy_score(participant_pred_df["y"], participant_pred_df["pred_median"])

# ROC_AUC requires both classes to be present in the combined held-out participant predicitions
try:
    overall_p_auc_mean = roc_auc_score(participant_pred_df["y"], participant_pred_df["prob_mean"])
except ValueError:
    overall_p_auc_mean = np.nan

try:
    overall_p_auc_median = roc_auc_score(participant_pred_df["y"], participant_pred_df["prob_median"])
except ValueError:
    overall_p_auc_median = np.nan

print("\nFixed-epoch LOPO participant-level results:")
print(f"Accuracy (mean agg):   {overall_p_acc_mean:.4f}")
print(f"ROC-AUC (mean agg):    {overall_p_auc_mean:.4f}")
print(f"Accuracy (median agg): {overall_p_acc_median:.4f}")
print(f"ROC-AUC (median agg):  {overall_p_auc_median:.4f}")

print("\nParticipant predictions across all outer held-out folds:")
print(participant_pred_df.sort_values(["y", "prob_mean"]))


Outer fold 1 | held-out participant: 67c26b6a6aed7a8612189b14
  Epoch 1/5 | train_loss=0.4632
  Epoch 2/5 | train_loss=0.4435
  Epoch 3/5 | train_loss=0.4159
  Epoch 4/5 | train_loss=0.3525
  Epoch 5/5 | train_loss=0.3362
Outer-train participant counts -> pos: 18, neg: 9


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



Outer fold 2 | held-out participant: 67c26fde6aed7a861218a3cb
  Epoch 1/5 | train_loss=0.5138
  Epoch 2/5 | train_loss=0.4847
  Epoch 3/5 | train_loss=0.4369
  Epoch 4/5 | train_loss=0.3969
  Epoch 5/5 | train_loss=0.3375
Outer-train participant counts -> pos: 17, neg: 10


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



Outer fold 3 | held-out participant: 67c27401f112be75fec84544
  Epoch 1/5 | train_loss=0.5081
  Epoch 2/5 | train_loss=0.4761
  Epoch 3/5 | train_loss=0.4254
  Epoch 4/5 | train_loss=0.3669
  Epoch 5/5 | train_loss=0.3272
Outer-train participant counts -> pos: 17, neg: 10


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



Outer fold 28 | held-out participant: 681bc78d7f0a181a7e96a598
  Epoch 1/5 | train_loss=0.5081
  Epoch 2/5 | train_loss=0.4661
  Epoch 3/5 | train_loss=0.4320
  Epoch 4/5 | train_loss=0.3688
  Epoch 5/5 | train_loss=0.3777
Outer-train participant counts -> pos: 17, neg: 10


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



Outer fold summary:
    outer_fold      held_out_participant  fixed_epochs  final_train_loss  \
0            1  67c26b6a6aed7a8612189b14             5          0.336160   
1            2  67c26fde6aed7a861218a3cb             5          0.337529   
2            3  67c27401f112be75fec84544             5          0.327228   
3            4  67c28656c6360e2457d02f60             5          0.401753   
4            5  67c65ae75af132787b58df9a             5          0.344310   
5            6  67c65b5db393e8e43f8a87bc             5          0.275942   
6            7  67c660bb89c7d2ddeb14ebf0             5          0.400648   
7            8  67c6688bf0dce1020cccb08b             5          0.383178   
8            9  67c777d88cb8fc5788663844             5          0.342883   
9           10  67c77911ff18ef1bd5d4ca8a             5          0.380345   
10          11  67c77b7fbe53b42f18bcb28c             5          0.412555   
11          12  67c77c008cb8fc57886640fb             5          0.3

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


## Better Model